In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder
import warnings

# Hide unnecessary warnings
warnings.filterwarnings('ignore')

start_time = time.time()

# ==========================================
# PHASE 1: EFFICIENT DATA LOADING & PREP
# ==========================================
print("1. Loading Data...")
train_df = pd.read_csv('train.csv', engine='pyarrow')
test_df = pd.read_csv('test.csv', engine='pyarrow')

# Downcast floats to save memory
train_df['demand'] = train_df['demand'].astype('float32')
if 'Temperature' in train_df.columns:
    train_df['Temperature'] = train_df['Temperature'].astype('float32')
    test_df['Temperature'] = test_df['Temperature'].astype('float32')

# Fill missing categorical data
categorical_features = ['geohash', 'timestamp', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
for col in categorical_features:
    train_df[col] = train_df[col].fillna('Missing').astype(str)
    test_df[col] = test_df[col].fillna('Missing').astype(str)

# ==========================================
# PHASE 2: EXPLORATORY DATA ANALYSIS (EDA)
# ==========================================
print("\n2. Running Fast EDA...")
print(f"Unique Geohashes: {train_df['geohash'].nunique()}")
print(f"Unique Timestamps: {train_df['timestamp'].nunique()}")

# ==========================================
# PHASE 3: MODEL PREP & ENCODING (CRITICAL FOR RF)
# ==========================================
print("\n3. Preparing Model Features...")
X = train_df.drop(columns=['Index', 'demand'], errors='ignore')
y = train_df['demand']
X_test = test_df.drop(columns=['Index'], errors='ignore')

# Random Forest requires numbers! We must encode the text categories.
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[categorical_features] = encoder.fit_transform(X[categorical_features])
X_test[categorical_features] = encoder.transform(X_test[categorical_features])

# Handle any missing numerical data (like missing Temperatures)
X = X.fillna(X.median())
X_test = X_test.fillna(X.median())

# Split 20% of data to check efficiency
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# PHASE 4: TRAINING RANDOM FOREST
# ==========================================
print("\n4. Training Random Forest (Please wait, using all CPU cores)...")
# n_jobs=-1 forces it to use maximum computer power
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# ==========================================
# PHASE 5: THE EFFICIENCY REVEAL
# ==========================================
val_predictions = model.predict(X_val)
r2 = r2_score(y_val, val_predictions)
rmse = np.sqrt(mean_squared_error(y_val, val_predictions))

print("\n" + "="*50)
print(f"🎯 RANDOM FOREST EFFICIENCY: {r2:.4f}")
print(f"   (This means your model is {r2 * 100:.2f}% successful)")
print(f"📉 RMSE (Average Error Margin):  {rmse:.4f}")
print("="*50 + "\n")

# ==========================================
# PHASE 6: FINAL PREDICTIONS
# ==========================================
print("Retraining on 100% of data for maximum test accuracy...")
model.fit(X, y) # Let it learn from the 20% we hid earlier!

print("Predicting on test.csv...")
test_predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Index': test_df['Index'], 
    'demand': test_predictions
})

submission.to_csv('randomforest_submission.csv', index=False)
print(f"Done! Saved to 'randomforest_submission.csv' in {time.time() - start_time:.2f} seconds.")

# ==========================================
# PHASE 7: VISUALIZATIONS
# ==========================================
plt.figure(figsize=(10, 5))
train_df['demand'].hist(bins=50, color='skyblue', edgecolor='black')
plt.title('Distribution of Demand')
plt.xlabel('Demand')
plt.ylabel('Frequency')
plt.show()